# GUS01C: GeoTERYT Database

This notebook demonstrates the GeoTERYT database system for Polish administrative divisions.

## Overview
- Build database from harmonized TERYT data (from GUS01A)
- Load geometries from shapefiles (from GUS01B)
- Search and query functionality
- Geometry operations (merging, best geometry selection)
- Overlay operations (pre-1999 voivodeships)

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
            return p
    return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
geospatial_root = repo_root.parent.parent / 'Data' / 'Geospatial'
geometry_root = repo_root.parent.parent / 'Data' / 'Geospatial' / 'geometry'
gus_root = Path(os.getcwd()).parent.parent.parent.parent / "Data" / "GUS"

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

# Import local toolkit
import local_utility_functions as luf
import geoTERYT_db as gtdb

print(f"Repo root: {repo_root}")
print(f"GUS root: {gus_root}")
print(f"Geometry root: {geometry_root}")

## 1. Load Harmonized TERYT Data

Load the mega DataFrame created in GUS01A notebook.

In [ ]:
# Load the saved mega_df from GUS01A/GUS01B
mega_df = pd.read_csv(geospatial_root / "teryt_df_backup.csv", encoding="utf-8")

# Ensure proper formatting
v_tostring = np.vectorize(lambda x: str(x))
v_zero_pad = np.vectorize(lambda x, n: str(x).zfill(n))
mega_df['id'] = v_zero_pad(v_tostring(mega_df['id']), 7)

print(f"Loaded mega_df: {len(mega_df)} rows")
print(f"Years: {mega_df['year'].min()} - {mega_df['year'].max()}")
print(f"Columns: {list(mega_df.columns)}")
mega_df.head()

## 2. Load Geometry Files

In [ ]:
import re

# Scan for geometry files
geometry_files = [f for f in os.listdir(geometry_root) if not f.startswith('.')]
geometry_paths = [geometry_root / f for f in geometry_files]

shape_files = []
for i in range(len(geometry_paths)):
    if geometry_paths[i].is_dir():
        dir_contents = os.listdir(geometry_paths[i])
        for file in dir_contents:
            if file.endswith('.shp'):
                shape_files.append(geometry_paths[i] / file)

shape_files.sort()
# Filter to gmina-level files
shape_files = [f for f in shape_files if 'Obszary' in f.name or 'gmin' in f.name]

print("Found shapefiles:")
for f in shape_files:
    print(f"  {f.parent.name}/{f.name}")

In [ ]:
# Load all shapefiles into a dictionary
gdf_list = {}
for file in shape_files:
    key = str(file.parent)[-4:] + "_" + str(file.stem)
    try:
        gdf = gpd.read_file(file)
        gdf_list[key] = gdf
        print(f"Loaded {key}: {len(gdf)} features")
    except Exception as e:
        print(f"Error loading {key}: {e}")

print(f"\nTotal loaded: {len(gdf_list)} GeoDataFrames")

In [ ]:
# Prepare GeoDataFrames: standardize teryt column and CRS
for key in gdf_list:
    print(f"Processing {key}...")
    
    # Extract year from key
    match = re.search(r'\d{4}', key)
    if not match:
        continue
    year = int(match.group(0))
    
    # Standardize column names to lowercase
    gdf_list[key].columns = [col.lower() for col in gdf_list[key].columns]
    
    # Convert CRS to EPSG:2180
    if gdf_list[key].crs is None:
        gdf_list[key] = gdf_list[key].set_crs('EPSG:2180')
    else:
        gdf_list[key] = gdf_list[key].to_crs('EPSG:2180')
    
    # Standardize TERYT column
    if year > 2012:
        gdf_list[key]['teryt'] = gdf_list[key]['jpt_kod_je']
    else:
        # Older files use 'obszar' with NUTS-like codes
        v_NCtT = np.vectorize(luf.nuts_code_to_teryt)
        v_tostring = np.vectorize(lambda x: str(x))
        gdf_list[key]['teryt'] = v_NCtT(v_tostring(gdf_list[key]['obszar'].values))
    
    # Sort by teryt
    gdf_list[key] = gdf_list[key].sort_values(by='teryt').reset_index(drop=True)
    
print("\nAll GeoDataFrames processed!")

In [ ]:
# For older geometries (<=2012), dissolve sub-units (RODZ 4,5) into main gmina
for key in gdf_list:
    match = re.search(r'\d{4}', key)
    if not match:
        continue
    year = int(match.group(0))
    
    if year <= 2012:
        print(f"Dissolving sub-units for {key}...")
        
        # Fix invalid geometries
        invalid_mask = ~gdf_list[key].is_valid
        if invalid_mask.any():
            print(f"  Fixing {invalid_mask.sum()} invalid geometries...")
            gdf_list[key].loc[invalid_mask, 'geometry'] = gdf_list[key].loc[invalid_mask, 'geometry'].buffer(0)
        
        # Create short teryt (replace last digit with '0')
        gdf_list[key]['teryt_short'] = gdf_list[key]['teryt'].str[:-1] + '0'
        gdf_list[key] = gdf_list[key].dissolve(by='teryt_short', as_index=False)
        gdf_list[key]['teryt'] = gdf_list[key]['teryt_short']
        gdf_list[key] = gdf_list[key].drop(columns=['teryt_short'])
        gdf_list[key] = gdf_list[key].sort_values(by='teryt').reset_index(drop=True)
        
        print(f"  Result: {len(gdf_list[key])} features")

print("\nDissolution complete!")

## 3. Build the GeoTERYT Database

In [ ]:
# Create and build the database
db = gtdb.GeoTERYTDatabase()
db.build_from_harmonized(mega_df, verbose=True)

In [ ]:
# Load geometries
db.load_geometries(gdf_list, teryt_column='teryt', verbose=True)

In [ ]:
# Print database summary
db.print_summary()

## 4. Search and Query Examples

In [ ]:
# Search by TERYT ID
teryt_id = '1461011'  # Example gmina
info = db.get_unit_info(teryt_id)

if info:
    print(f"Unit info for {teryt_id}:")
    for k, v in info.items():
        print(f"  {k}: {v}")
else:
    print(f"No unit found with ID {teryt_id}")

In [ ]:
# Search by name
search_name = "Warszawa"
results = db.search_by_name(search_name, exact=False)

print(f"Found {len(results)} units matching '{search_name}':")
for r in results[:10]:  # Show first 10
    print(f"  {r.teryt_id}: {r.name} ({r.kind}, years {r.first_year}-{r.last_year})")

In [ ]:
# Get all divisions for a specific year
year = 2020
gminas_2020 = db.get_divisions_by_year(year, level=6, exclude_subdivisions=True)

print(f"Gminas in {year}: {len(gminas_2020)}")
print(f"\nFirst 5:")
for r in gminas_2020[:5]:
    print(f"  {r.teryt_id}: {r.name} ({r.kind})")

In [ ]:
# Get year statistics
for year in [1999, 2010, 2020, 2024]:
    stats = db.get_year_statistics(year)
    print(f"\nYear {year}:")
    print(f"  Total units: {stats['total_units']}")
    print(f"  By level: {stats['level_counts']}")
    print(f"  Units with geometry: {stats['units_with_geometry']}")

In [ ]:
# Get units that changed
changed_units = db.get_changed_units(year_from=2015, year_to=2020)

print(f"Units with changes between 2015-2020: {len(changed_units)}")
print("\nExamples:")
for r in changed_units[:5]:
    print(f"  {r.teryt_id}: {r.name}")
    for c in r.changes:
        if 2015 <= c.get('year', 0) <= 2020:
            print(f"    [{c['year']}] {c['description']}")

## 5. Geometry Operations

In [ ]:
# Convert to GeoDataFrame for a specific year
gdf_2020 = db.to_geodataframe(year=2020, level=6, exclude_subdivisions=True)

print(f"GeoDataFrame for 2020 gminas: {len(gdf_2020)} rows")
print(f"Columns: {list(gdf_2020.columns)}")
print(f"Rows with geometry: {gdf_2020.geometry.notna().sum()}")

gdf_2020.head()

In [ ]:
# Plot gminas with geometry
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
gdf_valid = gdf_2020[gdf_2020.geometry.notna()]
gdf_valid.plot(ax=ax, edgecolor='black', linewidth=0.1, facecolor='lightblue')
ax.set_title(f'Poland Gminas in 2020 ({len(gdf_valid)} with geometry)')
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Merge to voivodeship level
voivodeships_2020 = db.merge_to_level(year=2020, target_level=2)

print(f"Voivodeships (merged): {len(voivodeships_2020)}")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
voivodeships_2020.plot(ax=ax, edgecolor='black', linewidth=1, 
                        column='name', legend=True, legend_kwds={'loc': 'upper left', 'fontsize': 8})
ax.set_title('Poland Voivodeships 2020 (merged from gminas)')
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Merge to powiat level
powiats_2020 = db.merge_to_level(year=2020, target_level=5)

print(f"Powiats (merged): {len(powiats_2020)}")

# Plot
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
powiats_2020.plot(ax=ax, edgecolor='black', linewidth=0.3, facecolor='lightgreen')
ax.set_title(f'Poland Powiats 2020 ({len(powiats_2020)} powiats merged from gminas)')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## 6. Compare Years

In [ ]:
# Compare gmina counts across years
years_to_compare = [1999, 2005, 2010, 2015, 2020, 2024]
counts = []

for year in years_to_compare:
    stats = db.get_year_statistics(year)
    gminas = stats['level_counts'].get(6, 0)
    powiats = stats['level_counts'].get(5, 0)
    voivodeships = stats['level_counts'].get(2, 0)
    counts.append({
        'year': year,
        'gminas': gminas,
        'powiats': powiats,
        'voivodeships': voivodeships
    })

counts_df = pd.DataFrame(counts)
print(counts_df.to_string(index=False))

In [ ]:
# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(counts_df['year'], counts_df['gminas'], color='skyblue')
axes[0].set_title('Gminas per Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')

axes[1].bar(counts_df['year'], counts_df['powiats'], color='lightgreen')
axes[1].set_title('Powiats per Year')
axes[1].set_xlabel('Year')

axes[2].bar(counts_df['year'], counts_df['voivodeships'], color='salmon')
axes[2].set_title('Voivodeships per Year')
axes[2].set_xlabel('Year')

plt.tight_layout()
plt.show()

## 7. Export Database

In [ ]:
# Export to DataFrame
db_df = db.to_dataframe(include_geometry=False)
print(f"Exported {len(db_df)} records to DataFrame")
db_df.head()

In [ ]:
# Save to CSV
output_path = geospatial_root / "geoteryt_database.csv"
db_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"Saved to {output_path}")

In [ ]:
# Export specific year to GeoPackage (if geopandas supports it)
try:
    output_gpkg = geospatial_root / "gminas_2020.gpkg"
    db.export_to_geopackage(output_gpkg, year=2020, level=6)
    print(f"Exported 2020 gminas to {output_gpkg}")
except Exception as e:
    print(f"GeoPackage export not available: {e}")

## 8. Summary

The GeoTERYT database has been successfully built and tested. Key capabilities:

1. **Data Storage**: All administrative divisions from 1999-2024
2. **Historical Tracking**: Name changes, ID changes, structural changes
3. **Geometry Integration**: Multiple shapefile sources linked to records
4. **Search**: By TERYT ID, name, year, level, kind
5. **Geometry Operations**: Merge to higher levels, best geometry selection
6. **Export**: DataFrame, CSV, GeoPackage, Shapefile